# 第 1 周末练习 —— 技术问答解释器（OpenRouter + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段代码在干什么？」）
- **输出**：清晰的解释
- **额外**：用**流式（streaming）**一边生成一边更新 Markdown 显示

这是你在课程期间自己也能天天用的工具。本练习通过 **OpenRouter** 调用云端模型，再用 `ollama` Python 包装库对比本地模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(..., stream=True)` |
| `messages`（system / user） | system 定导师人设，user 放具体问题 |
| 流式输出 | `update_display` 逐块刷新 Markdown |
| 兼容端点 | OpenRouter：`base_url=https://openrouter.ai/api/v1` |
| 本地模型 | `ollama.chat(model=MODEL_LLAMA, ...)` |

## 怎么跑

1. 从上到下依次运行单元格
2. `.env` 准备 `OPEN_ROUTER_API_KEY`；本地路径需安装 `ollama` 包并拉取 `phi3:latest`（或按提示改模型名）
3. 可改 `question` 字符串后重跑后面的流式 / Ollama 单元格


In [14]:
# ========== 导入：环境变量、笔记本展示、OpenAI 兼容客户端 ==========

# 导入标准库 os：读环境变量（例如 OPEN_ROUTER_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程，避免把密钥写进代码
from dotenv import load_dotenv
# Markdown / display / update_display：流式时不断刷新同一块输出
from IPython.display import Markdown, display, update_display
# OpenAI 客户端类：这里会通过 base_url 指向 OpenRouter
from openai import OpenAI


In [ ]:
# ========== 常量：云端模型、本地模型、OpenRouter 基址 ==========

# 经 OpenRouter 调用的云端小模型名
MODEL_GPT = "gpt-4o-mini"
# 本地 Ollama 模型标签；须与本机已 pull 的名字一致
MODEL_LLAMA = "phi3:latest"
# OpenRouter 的 OpenAI 兼容 API 根路径（后面创建客户端时作 base_url）
openRouter_url = "https://openrouter.ai/api/v1"


In [ ]:
# ========== 环境：加载密钥并做一次粗检查 ==========

# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 读取 OpenRouter 密钥；变量名必须是 OPEN_ROUTER_API_KEY（与常见 OPENAI_API_KEY 不同）
openRouter_api_key = os.getenv("OPEN_ROUTER_API_KEY")
# 启发式检查：有值、以 sk- 开头、长度够——不能保证密钥一定有效
if openRouter_api_key and openRouter_api_key.startswith("sk-") and len(openRouter_api_key) > 10:
    print("API key looks good so far")
else:
    # 提示文案保持英文原样
    print("There might be a problem with your API key? Please check your environment variable!")


In [ ]:
# ========== 提问：改这里的三引号字符串就能问新问题 ==========
# 发给模型的 question 正文保持英文/原样（含示例代码里的注释与笔误），避免改变模型行为
question = """
Please explain what this code does and why:
def findCatalan(n):
  
    # 【注】Base case
    if n <= 1:
        return 1

    # 【注】catalan(n) is sum of catalan(i) * catalan(n-i-1)
    res = 0
    for i in range(n):
        res += findCatalan(i) * findCatalan(n - i - 1)

    return res


n = 6
res = findCatalan(n)
print(res)}
"""


In [ ]:
# ========== 组装 prompts 与 messages：system 定人设，user 塞具体问题 ==========
# system / user 字符串保持英文（影响回答风格，不翻译）
system_prompt = """
You are a helpful technical tutor who answers questions about provided code or technical programming questions, software engineering, 
data science and LLMs. And other related topics to computer science.

Note:
- If question didn't specify which language to be explained in, explain it in programming language it is written in.
- If question didn't specify what to explain, explain the whole code or the part that you think is most important
- Explain necessary concepts and terms. unless user ask deep explanation to whole question or on specific question
"""
# 在固定前缀后拼接上面的 question
user_prompt = "Please give a detailed explanation to the following question:\n\n" + question

# Chat Completions 标准 messages 列表：先 system 后 user
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


In [ ]:
# ========== 云端流式：经 OpenRouter 调 GPT，边收边刷新 Markdown ==========

# 创建客户端：base_url 指向 OpenRouter，api_key 用刚才读到的密钥
openai = OpenAI(base_url=openRouter_url,api_key=openRouter_api_key)
# stream=True：返回可迭代的增量 chunk，而不是一次性完整回复
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
# 累积完整文本，供每次 update_display 重绘
response = ""

# display_id=True：拿到可更新的显示句柄，避免每一块都新开一块输出
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    # delta.content 可能为 None；用 or "" 拼进缓冲区
    response += chunk.choices[0].delta.content or ""
    # 用同一 display_id 刷新为最新的完整 Markdown
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 本地对比：用 ollama Python 包装库问同一套 messages ==========

# 尝试导入 ollama；没有就降级跳过，避免整格崩溃
try:
    import ollama
    print("Ollama package ready. Run 'ollama pull gemma3:270m' (or phi3) if you haven't already.")
except ImportError:
    ollama = None
    print("Install ollama package for Llama: pip install ollama")

# 仅当包可用时才调用本地模型
if ollama is not None:
    # 非流式 chat：一次返回完整 message
    response_llama = ollama.chat(model=MODEL_LLAMA, messages=messages)
    # 从返回字典取出助手文本
    reply = response_llama["message"]["content"]
    # 用 Markdown 展示本地模型回答，便于和上面 GPT 流式结果对比
    display(Markdown(reply))
else:
    # 跳过提示保持英文原样
    print("Skipping Llama: ollama package not installed.")
